In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from utils import aproximar_limite_valido

In [23]:
df = pd.read_csv("../data/comprovantes_pix_10000_anomalias.csv", sep=";")

In [24]:
df['Anomalia'].value_counts()

Anomalia
0    9900
1     100
Name: count, dtype: int64

In [25]:
df.agg(['min', 'max'])

,EndToEndId,DataHora,Valor,Moeda,Pagador_Nome,Pagador_CPF_CNPJ,Pagador_Banco,Recebedor_Nome,Recebedor_CPF_CNPJ,Recebedor_Banco,ChavePix_Utilizada,TipoChave,Descricao,Status,Anomalia
min,00032a00-5360-40da-aa43-ea59ce48593e,2025-02-30 25:61:00,0.00,BRL,Agatha Almeida,000.000.000-00,BTG Pactual,Agatha Araújo,10.127.878/0001-59,BTG Pactual,+551091035-2412,CNPJ,Pagamento referente ao serviço 1,Concluída,0
max,ffffad37-8e79-4cd8-a1b6-b05d20bc5b5e,2025-08-20 15:44:56,4999.79,BRL,Yuri da Rosa,999.790.314-14,Santander Brasil,Yuri das Neves,999.704.956-96,Santander Brasil,user9@email.com,Telefone,Pagamento referente ao serviço 999,Pendente,1


In [26]:
df.isnull().sum()

EndToEndId            0
DataHora              0
Valor                 0
Moeda                 0
Pagador_Nome          0
Pagador_CPF_CNPJ      0
Pagador_Banco         0
Recebedor_Nome        0
Recebedor_CPF_CNPJ    0
Recebedor_Banco       0
ChavePix_Utilizada    0
TipoChave             0
Descricao             0
Status                0
Anomalia              0
dtype: int64

In [27]:
df['DataHora_Tratada'] = df['DataHora'].apply(aproximar_limite_valido)

In [28]:
# O .dt.hour e .dt.dayofweek roubam apenas a hora e o dia da semana (onde 0 é segunda e 6 é domingo).
# função lambda para dizer: Se o dia for 5 (sábado) ou 6 (domingo), marque 1 (É fim de semana). Senão, marque 0.
df['Hora'] = df['DataHora_Tratada'].dt.hour
df['DiaDaSemana'] = df['DataHora_Tratada'].dt.dayofweek
df['FimDeSemana'] = df['DiaDaSemana'].apply(lambda x: 1 if x >= 5 else 0)

# Horário comercial ganha 1 apenas se for entre 8h e 18h e se não for fim de semana. Já a madrugada ganha 1 se a hora for de 0 a 5.
df['Horario_Comercial'] = df.apply(lambda row: 1 if (8 <= row['Hora'] <= 18) and (row['FimDeSemana'] == 0) else 0, axis=1)
df['Madrugada'] = df['Hora'].apply(lambda x: 1 if 0 <= x <= 5 else 0)

# Isola os dias de pico bancário (salários e vales). 
df['Dia_do_Mes'] = df['DataHora_Tratada'].dt.day
df['Dia_de_Pagamento'] = df['Dia_do_Mes'].apply(lambda x: 1 if x in [5, 6, 7, 20, 21, 22] else 0)

# Faz uma divisão por 1 (usando o %). Se o resto for zero, significa que não tem centavos (ex: R$ 500,00 redondo). 
df['Valor_Redondo'] = df['Valor'].apply(lambda x: 1 if x % 1 == 0 else 0)

df['Status_Pendente'] = df['Status'].apply(lambda x: 1 if x == 'Pendente' else 0)

# Compara se o banco de quem mandou o Pix é igual ao banco de quem recebeu. Fraudes frequentemente envolvem contas laranjas de bancos diferentes.
df['Mesmo_Banco'] = (df['Pagador_Banco'] == df['Recebedor_Banco']).astype(int)

In [29]:
df.drop(columns=['Descricao', 'DataHora', 'Moeda', 'Pagador_Nome', 'Pagador_CPF', 'Pagador_CPF_CNPJ', 'Pagador_Banco', 'Recebedor_Banco' ,'Recebedor_Nome', 'Recebedor_CPF_CNPJ', 'Recedor_Banco', 'ChavePix_Utilizada', 'Descricap'], inplace=True, errors='ignore')

In [30]:
df.head(2)

,EndToEndId,Valor,TipoChave,Status,Anomalia,DataHora_Tratada,Hora,DiaDaSemana,FimDeSemana,Horario_Comercial,Madrugada,Dia_do_Mes,Dia_de_Pagamento,Valor_Redondo,Status_Pendente,Mesmo_Banco
0,f094cb2c-2a73-463c-b60e-0c57262051e4,4658.86,Telefone,Concluída,0,2025-06-26 21:07:56,21,3,0,0,0,26,0,0,0,0
1,f0409769-741f-49bd-811a-3842cc8f54db,3184.72,Telefone,Pendente,0,2025-06-23 17:47:56,17,0,0,1,0,23,0,0,1,0


In [31]:
# Criando a transformação de Logaritmo (suaviza valores extremos)
df['Valor_Log'] = np.log1p(df['Valor'])

# Criando a Padronização (coloca dinheiro e tempo na mesma escala)
scaler = StandardScaler()
df[['Valor_std', 'Valor_Log_std', 'Hora_std']] = scaler.fit_transform(df[['Valor', 'Valor_Log', 'Hora']])

# Salvando a base agora 100% tratada
df.to_csv("../data/base_tratada.csv", index=False)

print("Base salva com sucesso! Todas as variáveis avançadas foram incluídas no CSV.")

Base salva com sucesso! Todas as variáveis avançadas foram incluídas no CSV.
